# Correct a polystyrene detector image

This example reads closed-shutter and open-shutter polystyrene Zarr stores, fits on every polystyrene frame except frame 67, corrects that held-out frame, and then row-sums the corrected image. If no cosmic ray is accepted, the image is returned unchanged.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from cosmic_ray_bayes import ImageCalibrator, read_image_zarr

In [ ]:
image_stores = {
    "dark": "../data/image/closed_shutter_2000ms.zarr",
    "polystyrene": "../data/image/open_shutter_polystyrene_2000ms.zarr",
}
images = read_image_zarr(image_stores, columns=range(1568, 1632))

dark_frames = images.xs(("dark", 2000.0), level=("experiment", "exposure_ms"))
polystyrene_frames = images.xs(
    ("polystyrene", 2000.0), level=("experiment", "exposure_ms")
)
calibration_frames = polystyrene_frames.drop(index=67, level="frame")
polystyrene_image = polystyrene_frames.xs(67, level="frame")
detector_columns = polystyrene_image.columns.to_numpy()

In [ ]:
calibrator = ImageCalibrator.fit(dark_frames, calibration_frames)
corrected_image = calibrator.correct(polystyrene_image)

In [ ]:
measured_image = polystyrene_image.to_numpy()
removed_charge = measured_image - corrected_image
display_limits = np.percentile(measured_image, (1, 99))
image_extent = (
    detector_columns[0] - 0.5,
    detector_columns[-1] + 0.5,
    -0.5,
    measured_image.shape[0] - 0.5,
)
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for axis, title, values in zip(
    axes[:2],
    ("measured polystyrene", "corrected polystyrene"),
    (measured_image, corrected_image),
    strict=True,
):
    image = axis.imshow(
        values,
        origin="lower",
        aspect="auto",
        cmap="viridis",
        vmin=display_limits[0],
        vmax=display_limits[1],
        extent=image_extent,
    )
    axis.set_title(title)
    fig.colorbar(image, ax=axis, shrink=0.75, label="ADU")
removed = axes[2].imshow(
    removed_charge,
    origin="lower",
    aspect="auto",
    cmap="magma",
    extent=image_extent,
)
axes[2].set_title("removed charge")
fig.colorbar(removed, ax=axes[2], shrink=0.75, label="ADU")
for axis in axes:
    axis.set_xlabel("detector column")
axes[0].set_ylabel("detector row")
plt.show()

In [ ]:
measured_row_sum = polystyrene_image.sum(axis=0).to_numpy()
corrected_row_sum = corrected_image.sum(axis=0)
removed_row_sum = measured_row_sum - corrected_row_sum

fig, axes = plt.subplots(
    2,
    1,
    figsize=(10, 5),
    sharex=True,
    constrained_layout=True,
    gridspec_kw={"height_ratios": (3, 1)},
)
axes[0].plot(detector_columns, measured_row_sum, label="measured", linewidth=1)
axes[0].plot(detector_columns, corrected_row_sum, label="corrected", linewidth=1)
axes[0].set_title("Row-summed polystyrene frame 67")
axes[0].set_ylabel("row-summed ADU")
axes[0].legend()
axes[1].plot(detector_columns, removed_row_sum, color="tab:red", linewidth=1)
axes[1].set_xlabel("detector column")
axes[1].set_ylabel("removed (ADU)")
plt.show()

`corrected_image` and its row-summed detector segment, `corrected_row_sum`, are ready for downstream analysis. Every detector value not corrected by the algorithm is copied from the measured input.